# Step 1 : Groupby - Average

In [1]:
# Necessary Libraries
import pandas as pd
import os
from datetime import datetime, time, timedelta
import math
import numpy as np

## Data Awal

### Data Reel

In [2]:
# Pipeline for Data Reel
def merge_reel_data(file_list):
    dataframes = []
    
    for file in file_list:
        if os.path.exists(file):
            df = pd.read_excel(file)
            dataframes.append(df)
            print(f"Berhasil memuat: {file} | Shape: {df.shape}")
        else:
            print(f"GAGAL MEMUAT: {file} tidak ditemukan di direktori.")
            
    if not dataframes:
        raise ValueError("Pipeline dihentikan. Tidak ada satupun file yang valid untuk digabungkan.")
        
    master_reel = pd.concat(dataframes, ignore_index=True)
    
    return master_reel

In [3]:
# Eksekusi Pipeline
daftar_file_reel = [
    "../Data Reel/data reel pm17 0326.xlsx",
    "../Data Reel/data reel pm17 0426.xlsx"
]

reel_pm17 = merge_reel_data(daftar_file_reel)
reel_pm17.head()

Berhasil memuat: ../Data Reel/data reel pm17 0326.xlsx | Shape: (678, 13)
Berhasil memuat: ../Data Reel/data reel pm17 0426.xlsx | Shape: (611, 13)


,Time,Tanggal,Grade,Shift,Reel,Bw,Thickness,MDT,CDT,MDWT,MDS,Brightness,Insp. Status
0,7.0,01.03.26,T 15.1.Recycle,1,72,15.32,0.85,974,430,114,18,80.3,Acc Sotiss
1,8.1,01.03.26,T 15.1.Recycle,1,73,15.62,0.84,954,466,116,17,80.7,Acc Sotiss
2,9.3,01.03.26,T 15.1.Recycle,1,74,15.30,0.83,981,457,109,17,80.1,Acc Sotiss
3,10.4,01.03.26,T 15.1.Recycle,1,75,15.42,0.84,1060,442,118,17,80.7,Acc Sotiss
4,11.5,01.03.26,T 15.1.Recycle,1,76,15.56,0.80,959,448,107,19,80.9,Acc Sotiss


### Params PM

In [4]:
# Pipeline for Params PM
def load_and_standardize(file_path):
    _, ext = os.path.splitext(file_path)
    
    if ext.lower() in ['.xlsx', '.xls']:
        df = pd.read_excel(file_path)
    elif ext.lower() == '.csv':
        with open(file_path, encoding='utf-16') as f:
            raw_header = f.readline().strip()
            if raw_header.startswith('"') and raw_header.endswith('"') and '""' in raw_header:
                header = raw_header.strip('"').replace('""', '').split(';')
            else:
                header = [col.strip().strip('"') for col in raw_header.split(';')]
            
            df = pd.read_csv(f, delimiter=';', names=header)
    else:
        raise ValueError(f"Format file tidak dikenali: {ext}")

    # Standarisasi Header
    df.columns = df.columns.str.strip()

    rename_map = {
        'Speed Yankee'   : 'Yankee Speed',
        'Preasure Yankee': 'Yankee Pressure'
    }
    df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns}, inplace=True)

    # Validasi Waktu sebagai Kunci Utama
    df.dropna(subset=['Time'], inplace=True)
    df['Time'] = pd.to_datetime(df['Time'], dayfirst=True, format='mixed', errors='coerce')

    return df

def build_master_pipeline(file_list):
    dataframes = []
    
    for file in file_list:
        try:
            df = load_and_standardize(file)
            dataframes.append(df)
        except FileNotFoundError:
            print(f"Peringatan: File {file} tidak ditemukan. Dilewati.")
            
    if not dataframes:
        raise ValueError("Tidak ada data yang berhasil dimuat.")

    master_df = pd.concat(dataframes, ignore_index=True)
    master_df.drop_duplicates(subset=['Time'], keep='last', inplace=True)
    master_df.sort_values('Time', inplace=True)
    
    return master_df.reset_index(drop=True)

In [5]:
# Eksekusi Pipeline
file_sources = [
    '../PM Params/28042026_PM17.csv',
    '../PM Params/04052026_PM17.csv',
    '../PM Params/11052026_PM17.csv'
]
raw17 = build_master_pipeline(file_sources)
raw17.info()

<class 'pandas.DataFrame'>
RangeIndex: 27920 entries, 0 to 27919
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   Time               27920 non-null  datetime64[us]
 1   Yankee Speed       27920 non-null  float64       
 2   Pope Reel Speed    27920 non-null  float64       
 3   Yankee Pressure    27920 non-null  float64       
 4   Stock Flow         27920 non-null  float64       
 5   Stock Consistency  27920 non-null  float64       
 6   Flow Coating       27920 non-null  float64       
 7   Flow Release       27920 non-null  float64       
 8   Jet Wire Ratio     27920 non-null  float64       
 9   Load KWH Refiner   27920 non-null  float64       
dtypes: datetime64[us](1), float64(9)
memory usage: 2.1 MB


In [6]:
raw17.head()

,Time,Yankee Speed,Pope Reel Speed,Yankee Pressure,Stock Flow,Stock Consistency,Flow Coating,Flow Release,Jet Wire Ratio,Load KWH Refiner
0,2026-04-20 13:48:10,869.976318,756.61,7.20,1537.16,3.32,28.06,22.68,0.98,194.41
1,2026-04-20 13:49:10,870.064514,756.76,7.21,1538.49,3.33,28.06,22.68,0.98,194.24
2,2026-04-20 13:50:10,869.800110,756.76,7.25,1538.80,3.33,28.06,22.68,0.98,195.62
3,2026-04-20 13:51:10,869.711975,756.76,7.29,1539.85,3.34,28.06,22.68,0.98,195.17
4,2026-04-20 13:52:10,869.888123,756.98,7.29,1540.44,3.34,28.07,22.68,0.98,194.56


In [7]:
raw17['Yankee Speed'] = raw17['Yankee Speed'].astype(float)
raw17.info()

<class 'pandas.DataFrame'>
RangeIndex: 27920 entries, 0 to 27919
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   Time               27920 non-null  datetime64[us]
 1   Yankee Speed       27920 non-null  float64       
 2   Pope Reel Speed    27920 non-null  float64       
 3   Yankee Pressure    27920 non-null  float64       
 4   Stock Flow         27920 non-null  float64       
 5   Stock Consistency  27920 non-null  float64       
 6   Flow Coating       27920 non-null  float64       
 7   Flow Release       27920 non-null  float64       
 8   Jet Wire Ratio     27920 non-null  float64       
 9   Load KWH Refiner   27920 non-null  float64       
dtypes: datetime64[us](1), float64(9)
memory usage: 2.1 MB


In [8]:
# Create Categories PM Stop/Run
raw17['PM_stop'] = np.where(raw17['Pope Reel Speed'] == 0, "stop", "run")

# Create Coating/(Area.Min) Feature
raw17['Coating/(Area.Min)'] = ((raw17['Flow Coating'] * 60) / (raw17['Yankee Speed'] * 3050))

In [9]:
raw17.info()

<class 'pandas.DataFrame'>
RangeIndex: 27920 entries, 0 to 27919
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   Time                27920 non-null  datetime64[us]
 1   Yankee Speed        27920 non-null  float64       
 2   Pope Reel Speed     27920 non-null  float64       
 3   Yankee Pressure     27920 non-null  float64       
 4   Stock Flow          27920 non-null  float64       
 5   Stock Consistency   27920 non-null  float64       
 6   Flow Coating        27920 non-null  float64       
 7   Flow Release        27920 non-null  float64       
 8   Jet Wire Ratio      27920 non-null  float64       
 9   Load KWH Refiner    27920 non-null  float64       
 10  PM_stop             27920 non-null  str           
 11  Coating/(Area.Min)  27408 non-null  float64       
dtypes: datetime64[us](1), float64(10), str(1)
memory usage: 2.6 MB


#### Filter Out

In [10]:
# Filter Out Data
raw17 = raw17[(raw17['Yankee Speed'] >= 500) & (raw17['Pope Reel Speed'] >= 400)]

In [11]:
raw17.info()

<class 'pandas.DataFrame'>
Index: 24921 entries, 0 to 27919
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   Time                24921 non-null  datetime64[us]
 1   Yankee Speed        24921 non-null  float64       
 2   Pope Reel Speed     24921 non-null  float64       
 3   Yankee Pressure     24921 non-null  float64       
 4   Stock Flow          24921 non-null  float64       
 5   Stock Consistency   24921 non-null  float64       
 6   Flow Coating        24921 non-null  float64       
 7   Flow Release        24921 non-null  float64       
 8   Jet Wire Ratio      24921 non-null  float64       
 9   Load KWH Refiner    24921 non-null  float64       
 10  PM_stop             24921 non-null  str           
 11  Coating/(Area.Min)  24921 non-null  float64       
dtypes: datetime64[us](1), float64(10), str(1)
memory usage: 2.5 MB


In [12]:
# Delete before-after 0 values in 'Pope Reel Speed'
raw17 = raw17.reset_index(drop=True)
is_zero = np.isclose(raw17['Pope Reel Speed'], 0, atol=1e-5)
mask_to_drop = pd.Series(is_zero).rolling(window=11, center=True, min_periods=1).max().astype(bool)
df_clean = raw17[~mask_to_drop].copy()

In [13]:
df_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 24921 entries, 0 to 24920
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   Time                24921 non-null  datetime64[us]
 1   Yankee Speed        24921 non-null  float64       
 2   Pope Reel Speed     24921 non-null  float64       
 3   Yankee Pressure     24921 non-null  float64       
 4   Stock Flow          24921 non-null  float64       
 5   Stock Consistency   24921 non-null  float64       
 6   Flow Coating        24921 non-null  float64       
 7   Flow Release        24921 non-null  float64       
 8   Jet Wire Ratio      24921 non-null  float64       
 9   Load KWH Refiner    24921 non-null  float64       
 10  PM_stop             24921 non-null  str           
 11  Coating/(Area.Min)  24921 non-null  float64       
dtypes: datetime64[us](1), float64(10), str(1)
memory usage: 2.4 MB


In [14]:
df_clean.head()

,Time,Yankee Speed,Pope Reel Speed,Yankee Pressure,Stock Flow,Stock Consistency,Flow Coating,Flow Release,Jet Wire Ratio,Load KWH Refiner,PM_stop,Coating/(Area.Min)
0,2026-04-20 13:48:10,869.976318,756.61,7.20,1537.16,3.32,28.06,22.68,0.98,194.41,run,0.000635
1,2026-04-20 13:49:10,870.064514,756.76,7.21,1538.49,3.33,28.06,22.68,0.98,194.24,run,0.000634
2,2026-04-20 13:50:10,869.800110,756.76,7.25,1538.80,3.33,28.06,22.68,0.98,195.62,run,0.000635
3,2026-04-20 13:51:10,869.711975,756.76,7.29,1539.85,3.34,28.06,22.68,0.98,195.17,run,0.000635
4,2026-04-20 13:52:10,869.888123,756.98,7.29,1540.44,3.34,28.07,22.68,0.98,194.56,run,0.000635


## Preparation

### Preparation Process

#### Parameter PM

In [15]:
shift1_start = time(7, 0, 1)
shift1_end   = time(15, 0, 0)
shift2_start = time(15, 0, 1)
shift2_end   = time(23, 0, 0)
def assign_shift(t):
    if shift1_start <= t <= shift1_end:
        return 'Shift 1'
    elif shift2_start <= t <= shift2_end:
        return 'Shift 2'
    else:
        return 'Shift 3'

In [16]:
urutan_params_pm = [
    'Date', 'Time', 'Shift', 'Join_Key', 'Timestamp',
    'Creping', 'Yankee Speed', 'Pope Reel Speed',
    'Yankee Pressure', 'Stock Flow', 'Stock Consistency',
    'Flow Coating', 'Flow Release', 'Jet Wire Ratio',
    'Load KWH Refiner','Key_Date', 'PM_stop', 'Coating/(Area.Min)'
]

In [17]:
def preprocess_pm(df_input):
    df = df_input.copy()
    df = df.drop(columns=['Time'])
    df.columns = df.columns.str.strip()
    df.insert(0, 'Creping', (df['Yankee Speed'] - df['Pope Reel Speed']) * 100 / df['Yankee Speed'])
    def extract_time(x):
        if isinstance(x, str):
            return datetime.strptime(x.split(' ')[1], '%H:%M:%S').time()
        else:  # sudah datetime/Timestamp
            return x.time()
    df.insert(0, 'Time', df_input['Time'].apply(extract_time))
    df['Shift'] = df['Time'].apply(assign_shift)
    
    def extract_date(x):
        if isinstance(x, str):
            return x.split(' ')[0]
        else:
            return x.strftime('%d/%m/%y')
    df.insert(0, 'Date', df_input['Time'].apply(extract_date))
    df['Date'] = pd.to_datetime(df['Date'], format='%d/%m/%y')
    df['Key_Date'] = [df['Date'][index] - timedelta(days=1) 
                if
                df['Shift'][index] == 'Shift 3' 
                else df['Date'][index] 
                for index in range(len(df['Shift']))]
    df['Join_Key'] = df['Key_Date'].astype(str) + ' ' + df['Time'].astype(str) + ' ' + df['Shift']
    df['Timestamp'] = df['Date'].astype(str) + ' ' + df['Time'].astype(str)
    df['Shift'] = df['Shift'].str.extract(r'(\d+)').astype(int)
    df = df[urutan_params_pm]
    df.drop(columns=['Key_Date'], inplace=True)
    return df

#### Data Reel

In [18]:
def time_to_hms(val):
    s = str(val).strip()
    if s.lower() in {'', 'nan', 'none'}:
        return pd.NA

    # If already contains colon, parse parts directly
    if ':' in s:
        parts = s.split(':')
        h = int(parts[0])
        m = int(parts[1]) if len(parts) > 1 and parts[1] != '' else 0
        sec = int(parts[2]) if len(parts) > 2 and parts[2] != '' else 0

    # If contains dot, treat left as hours and right as minutes (common human shorthand)
    elif '.' in s:
        left, right = s.split('.', 1)
        if int(left) >= 24:
            return pd.NA  # Invalid hour value
        left = 0 if left == "24" else left  # Handle "24" as "00"
        h = int(left) if left != '' else 0

        # If right part is short (1 or 2 digits) treat it as minutes (e.g., "4.1" -> 4:01, "11.55" -> 11:55)
        if len(right) <= 2:
            m = int(right)
            sec = 0
        else:
            # If right part is longer, treat the whole value as a decimal hour (fallback)
            # e.g., "4.125" -> 4.125 hours -> convert fractional hour to minutes
            f = float(s)
            total_minutes = int(round((f - math.floor(f)) * 60))
            m = total_minutes
            sec = 0

    # No separator: treat as hours only (e.g., "6" -> 06:00:00)
    else:
        h = int(float(s))
        m = 0
        sec = 0

    # Normalize minutes >= 60 into hours
    if m >= 60:
        extra_h = m // 60
        h = (h + extra_h) % 24
        m = m % 60

    return f"{h:02d}:{m:02d}:{sec:02d}"

In [19]:
def preprocess_reel(df_input):
    df = df_input.copy()
    df['Time'] = df['Time'].apply(time_to_hms)
    df['Tanggal'] = pd.to_datetime(df['Tanggal'], format='%d.%m.%y')
    df = df.dropna(subset = ['Time']).reset_index(drop = True)
    df['Timestamp'] = df['Tanggal'].astype(str) + ' ' + df['Time'].astype(str)
    df['Timestamp'] = pd.to_datetime(df['Timestamp'], format='%Y-%m-%d %H:%M:%S')
    # Edit Kolom
    df = df.rename(columns={'Tanggal': 'Date'})
    first_cols = ['Date', 'Time', 'Shift', 'Timestamp']
    other_cols = [col for col in df.columns if col not in first_cols]
    df = df[first_cols + other_cols]
    return df

### Apply Preparation

#### Parameter PM

In [20]:
df_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 24921 entries, 0 to 24920
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   Time                24921 non-null  datetime64[us]
 1   Yankee Speed        24921 non-null  float64       
 2   Pope Reel Speed     24921 non-null  float64       
 3   Yankee Pressure     24921 non-null  float64       
 4   Stock Flow          24921 non-null  float64       
 5   Stock Consistency   24921 non-null  float64       
 6   Flow Coating        24921 non-null  float64       
 7   Flow Release        24921 non-null  float64       
 8   Jet Wire Ratio      24921 non-null  float64       
 9   Load KWH Refiner    24921 non-null  float64       
 10  PM_stop             24921 non-null  str           
 11  Coating/(Area.Min)  24921 non-null  float64       
dtypes: datetime64[us](1), float64(10), str(1)
memory usage: 2.4 MB


In [21]:
pm_17 = preprocess_pm(df_clean)
pm_17.head()

,Date,Time,Shift,Join_Key,Timestamp,Creping,Yankee Speed,Pope Reel Speed,Yankee Pressure,Stock Flow,Stock Consistency,Flow Coating,Flow Release,Jet Wire Ratio,Load KWH Refiner,PM_stop,Coating/(Area.Min)
0,2026-04-20,13:48:10,1,2026-04-20 13:48:10 Shift 1,2026-04-20 13:48:10,13.030966,869.976318,756.61,7.20,1537.16,3.32,28.06,22.68,0.98,194.41,run,0.000635
1,2026-04-20,13:49:10,1,2026-04-20 13:49:10 Shift 1,2026-04-20 13:49:10,13.022542,870.064514,756.76,7.21,1538.49,3.33,28.06,22.68,0.98,194.24,run,0.000634
2,2026-04-20,13:50:10,1,2026-04-20 13:50:10 Shift 1,2026-04-20 13:50:10,12.996102,869.800110,756.76,7.25,1538.80,3.33,28.06,22.68,0.98,195.62,run,0.000635
3,2026-04-20,13:51:10,1,2026-04-20 13:51:10 Shift 1,2026-04-20 13:51:10,12.987285,869.711975,756.76,7.29,1539.85,3.34,28.06,22.68,0.98,195.17,run,0.000635
4,2026-04-20,13:52:10,1,2026-04-20 13:52:10 Shift 1,2026-04-20 13:52:10,12.979614,869.888123,756.98,7.29,1540.44,3.34,28.07,22.68,0.98,194.56,run,0.000635


#### Data Reel

In [22]:
reel_pm17 = preprocess_reel(reel_pm17)
reel_pm17.head()

,Date,Time,Shift,Timestamp,Grade,Reel,Bw,Thickness,MDT,CDT,MDWT,MDS,Brightness,Insp. Status
0,2026-03-01,07:00:00,1,2026-03-01 07:00:00,T 15.1.Recycle,72,15.32,0.85,974,430,114,18,80.3,Acc Sotiss
1,2026-03-01,08:01:00,1,2026-03-01 08:01:00,T 15.1.Recycle,73,15.62,0.84,954,466,116,17,80.7,Acc Sotiss
2,2026-03-01,09:03:00,1,2026-03-01 09:03:00,T 15.1.Recycle,74,15.30,0.83,981,457,109,17,80.1,Acc Sotiss
3,2026-03-01,10:04:00,1,2026-03-01 10:04:00,T 15.1.Recycle,75,15.42,0.84,1060,442,118,17,80.7,Acc Sotiss
4,2026-03-01,11:05:00,1,2026-03-01 11:05:00,T 15.1.Recycle,76,15.56,0.80,959,448,107,19,80.9,Acc Sotiss


## Pipeline

In [23]:
# 1. MEMBACA DATA
df_reel = reel_pm17.copy()
df_params = pm_17.copy()

# 2. KONVERSI TIMESTAMP
df_reel['Timestamp'] = pd.to_datetime(df_reel['Timestamp'])
df_params['Timestamp'] = pd.to_datetime(df_params['Timestamp'])

# 3. SORT KEY
# REEL: Jam 00-06 ditambah 1 hari (karena di Excel tanggalnya mundur 1 hari dari params)
def create_sort_key_reel(ts):
    if ts.hour < 7:
        return ts + timedelta(days=1)
    return ts

df_reel['Sort_Key'] = df_reel['Timestamp'].apply(create_sort_key_reel)
df_params['Sort_Key'] = df_params['Timestamp']  # Params tidak perlu adjustment

# 4. FILTER BERDASARKAN SORT_KEY RANGE PARAMS
params_sort_min = df_params['Sort_Key'].min()
params_sort_max = df_params['Sort_Key'].max()

df_reel_filtered = df_reel[
    (df_reel['Sort_Key'] >= params_sort_min) & 
    (df_reel['Sort_Key'] <= params_sort_max)
].copy()

df_reel_filtered = df_reel_filtered.sort_values('Sort_Key').reset_index(drop=True)

# 5. DAFTAR VARIABEL
cols_to_avg = [
    'Creping', 'Yankee Speed', 'Pope Reel Speed', 'Yankee Pressure',
    'Stock Flow', 'Stock Consistency', 'Flow Coating', 'Flow Release',
    'Jet Wire Ratio', 'Load KWH Refiner', 'Coating/(Area.Min)'
]

# 6. LOOPING GROUPBY AVERAGE
results = []

for i in range(len(df_reel_filtered) - 1):
    start_time = df_reel_filtered['Timestamp'].iloc[i]
    end_time = df_reel_filtered['Timestamp'].iloc[i + 1]
    start_sort = df_reel_filtered['Sort_Key'].iloc[i]
    end_sort = df_reel_filtered['Sort_Key'].iloc[i + 1]
    
    mask = (df_params['Sort_Key'] >= start_sort) & (df_params['Sort_Key'] < end_sort)
    df_filtered = df_params.loc[mask]
    
    if len(df_filtered) == 0:
        continue
    
    row = {
        'Start_Time': start_time,
        'End_Time': end_time,
        'Data_Count': len(df_filtered)
    }
    
    for col in cols_to_avg:
        mean_val = df_filtered[col].mean()
        row[f'Mean_{col}'] = round(mean_val, 6) if pd.notna(mean_val) else None
    
    results.append(row)

# 7. HASIL
df_result = pd.DataFrame(results)

# 8. SIMPAN
# df_result.to_excel('grouby_params.xlsx', index=False)
print("\n✅ File disimpan: grouby_params.xlsx")


✅ File disimpan: grouby_params.xlsx


# Join Table

## Joining Df_Results and Reel Data

In [24]:
df_groupby = df_result.copy()
df_groupby.head()

,Start_Time,End_Time,Data_Count,Mean_Creping,Mean_Yankee Speed,Mean_Pope Reel Speed,Mean_Yankee Pressure,Mean_Stock Flow,Mean_Stock Consistency,Mean_Flow Coating,Mean_Flow Release,Mean_Jet Wire Ratio,Mean_Load KWH Refiner,Mean_Coating/(Area.Min)
0,2026-04-20 15:00:00,2026-04-20 16:02:00,59,13.607895,869.916566,751.539322,7.252712,1539.531356,3.321186,28.064576,22.680339,0.986102,202.135932,0.000635
1,2026-04-20 16:02:00,2026-04-20 17:37:00,95,13.696868,869.918799,750.767158,7.249474,1539.796737,3.320000,28.064737,22.679474,0.990000,196.412316,0.000635
2,2026-04-20 17:37:00,2026-04-20 18:55:00,78,13.641202,869.910785,751.244487,7.250385,1539.867179,3.319231,28.065000,22.679359,0.990000,194.285769,0.000635
3,2026-04-20 18:55:00,2026-04-20 20:02:00,67,12.508369,869.906601,761.095522,7.249552,1538.578358,3.319254,28.065672,22.679254,0.990000,187.730149,0.000635
4,2026-04-20 20:02:00,2026-04-20 21:31:00,89,12.094241,869.919879,764.709663,7.249888,1539.623483,3.321910,28.065169,22.679326,0.990000,189.772697,0.000635


In [25]:
df_reel = reel_pm17.copy()
df_reel.head()

,Date,Time,Shift,Timestamp,Grade,Reel,Bw,Thickness,MDT,CDT,MDWT,MDS,Brightness,Insp. Status
0,2026-03-01,07:00:00,1,2026-03-01 07:00:00,T 15.1.Recycle,72,15.32,0.85,974,430,114,18,80.3,Acc Sotiss
1,2026-03-01,08:01:00,1,2026-03-01 08:01:00,T 15.1.Recycle,73,15.62,0.84,954,466,116,17,80.7,Acc Sotiss
2,2026-03-01,09:03:00,1,2026-03-01 09:03:00,T 15.1.Recycle,74,15.30,0.83,981,457,109,17,80.1,Acc Sotiss
3,2026-03-01,10:04:00,1,2026-03-01 10:04:00,T 15.1.Recycle,75,15.42,0.84,1060,442,118,17,80.7,Acc Sotiss
4,2026-03-01,11:05:00,1,2026-03-01 11:05:00,T 15.1.Recycle,76,15.56,0.80,959,448,107,19,80.9,Acc Sotiss


In [26]:
# Create Join Key in df_groupby
df_groupby['Join_Key_Timestamp'] = df_groupby['End_Time'] #End_Time
# Create Join Key in df_reel
df_reel['Join_Key_Timestamp'] = df_reel['Timestamp']

In [27]:
# Join df_groupby with df_reel on Join_Key_Timestamp
df_joined = pd.merge(df_groupby, df_reel, left_on='Join_Key_Timestamp', right_on='Join_Key_Timestamp', how='inner')
df_joined.head()

,Start_Time,End_Time,Data_Count,Mean_Creping,Mean_Yankee Speed,Mean_Pope Reel Speed,Mean_Yankee Pressure,Mean_Stock Flow,Mean_Stock Consistency,Mean_Flow Coating,...,Grade,Reel,Bw,Thickness,MDT,CDT,MDWT,MDS,Brightness,Insp. Status
0,2026-04-20 15:00:00,2026-04-20 16:02:00,59,13.607895,869.916566,751.539322,7.252712,1539.531356,3.321186,28.064576,...,TW 22.1.Recycle,42,22.92,0.99,1935,1101,618,20,79.8,Acc Sotiss
1,2026-04-20 16:02:00,2026-04-20 17:37:00,95,13.696868,869.918799,750.767158,7.249474,1539.796737,3.320000,28.064737,...,TW 22.1.Recycle,43,22.92,1.02,1812,1059,509,20,79.1,Acc Sotiss
2,2026-04-20 17:37:00,2026-04-20 18:55:00,78,13.641202,869.910785,751.244487,7.250385,1539.867179,3.319231,28.065000,...,TW 22.1.Recycle,44,23.10,1.09,2001,1116,549,20,79.5,Acc Sotiss
3,2026-04-20 18:55:00,2026-04-20 20:02:00,67,12.508369,869.906601,761.095522,7.249552,1538.578358,3.319254,28.065672,...,TW 22.1.Recycle,45,22.74,1.07,2005,1038,585,18,79.9,Acc Sotiss
4,2026-04-20 20:02:00,2026-04-20 21:31:00,89,12.094241,869.919879,764.709663,7.249888,1539.623483,3.321910,28.065169,...,TW 22.1.Recycle,46,22.86,1.11,2172,1107,677,17,78.9,Acc Sotiss


## Joining df_joined with BB Table

### Data BB

In [28]:
# Pipeline for Data Reel
def merge_BB_data(file_list):
    dataframes = []
    
    for file in file_list:
        if os.path.exists(file):
            df = pd.read_excel(file, engine='openpyxl')
            dataframes.append(df)
            df.drop(columns=['Grade'], inplace=True)
            print(f"Berhasil memuat: {file} | Shape: {df.shape}")
        else:
            print(f"GAGAL MEMUAT: {file} tidak ditemukan di direktori.")
            
    if not dataframes:
        raise ValueError("Pipeline dihentikan. Tidak ada satupun file yang valid untuk digabungkan.")
        
    master_reel = pd.concat(dataframes, ignore_index=True)
    
    return master_reel

In [29]:
# Eksekusi Pipeline
file_sources = [
    "E:\Kuliah\Sun Paper Source\Efficiency\BB_Maret_PM17.xlsx",
    "E:\Kuliah\Sun Paper Source\Efficiency\BB_April_PM17.xlsx"
]
df_BB = merge_BB_data(file_sources)
df_BB.tail()

Berhasil memuat: E:\Kuliah\Sun Paper Source\Efficiency\BB_Maret_PM17.xlsx | Shape: (31, 10)
Berhasil memuat: E:\Kuliah\Sun Paper Source\Efficiency\BB_April_PM17.xlsx | Shape: (31, 8)


<>:3: SyntaxWarning: "\K" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\K"? A raw string is also an option.
<>:4: SyntaxWarning: "\K" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\K"? A raw string is also an option.
<>:3: SyntaxWarning: "\K" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\K"? A raw string is also an option.
<>:4: SyntaxWarning: "\K" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\K"? A raw string is also an option.
C:\Users\user\AppData\Local\Temp\ipykernel_14928\2724374265.py:3: SyntaxWarning: "\K" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\K"? A raw string is also an option.
  "E:\Kuliah\Sun Paper Source\Efficiency\BB_Maret_PM17.xlsx",
C:\Users\user\AppData\Local\Temp\ipykernel_14928\2724374265.py:4: SyntaxWarning: "\K" is an invalid escape sequen

,Date,GSM,Total NBKP,Total LBKP,Total BB Recycle,Sub Total Pulp+Broke,% NBKP,% LBKP,% BB Recycle,% Pulp+Broke
57,2026-04-27,"13,5",0.0,NaN,47643.860876,47643.860876,0.0,NaN,100.0,1.263516e+06
58,2026-04-28,"13,5",0.0,NaN,35329.923721,35329.923721,0.0,NaN,100.0,1.298846e+06
59,2026-04-29,"13,5",0.0,NaN,47453.335076,47453.335076,0.0,NaN,100.0,1.346299e+06
60,2026-04-30,"13,5/13/14",0.0,NaN,44018.612943,44018.612943,0.0,NaN,100.0,1.390318e+06
61,2026-05-01,0,0.0,NaN,0.000000,0.000000,0.0,NaN,0.0,1.390318e+06


df_BB = pd.read_excel("../Efficiency/BB_Maret_PM17.xlsx", engine='openpyxl')
df_BB.drop(columns=['Grade'], inplace=True)
df_BB.head()

### Last Join

In [30]:
# Standarisasi kolom "date" ke bentuk datetime
df_joined['Date'] = pd.to_datetime(df_joined['Date'])
df_BB['Date'] = pd.to_datetime(df_BB['Date'])

In [31]:
# Joining
df_final = pd.merge(df_joined, df_BB, on='Date', how='inner') #inner
df_final.info()

<class 'pandas.DataFrame'>
RangeIndex: 191 entries, 0 to 190
Data columns (total 38 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   Start_Time               191 non-null    datetime64[us]
 1   End_Time                 191 non-null    datetime64[us]
 2   Data_Count               191 non-null    int64         
 3   Mean_Creping             191 non-null    float64       
 4   Mean_Yankee Speed        191 non-null    float64       
 5   Mean_Pope Reel Speed     191 non-null    float64       
 6   Mean_Yankee Pressure     191 non-null    float64       
 7   Mean_Stock Flow          191 non-null    float64       
 8   Mean_Stock Consistency   191 non-null    float64       
 9   Mean_Flow Coating        191 non-null    float64       
 10  Mean_Flow Release        191 non-null    float64       
 11  Mean_Jet Wire Ratio      191 non-null    float64       
 12  Mean_Load KWH Refiner    191 non-null    float6

In [32]:
# Membaca file sebelumnya
df = df_final.copy()

# Cleaning - GSM
df['GSM'] = df['GSM'].astype(str)
df = df[~df['GSM'].str.contains('/', na=False)].copy()
df['GSM'] = df['GSM'].str.replace(',', '.')
df['GSM'] = df['GSM'].astype(float)
# Cleaning - MDWT
df['MDWT'] = pd.to_numeric(df['MDWT'], errors='coerce')
df = df.dropna(subset=['MDWT']).copy()

In [33]:
df['GSM'].tail(50)

116    13.5
117    13.5
118    13.5
119    13.5
120    13.5
121    13.5
122    13.5
123    13.5
124    13.5
125    13.5
126    13.5
127    13.5
128    13.5
129    13.5
130    13.5
131    13.5
132    13.5
133    13.5
134    13.5
135    13.5
136    13.5
137    13.5
138    13.5
139    13.5
140    13.5
141    13.5
142    13.5
143    13.5
144    13.5
145    13.5
146    13.5
147    13.5
148    13.5
149    13.5
150    13.5
151    13.5
152    13.5
153    13.5
154    13.5
155    13.5
156    13.5
157    13.5
158    13.5
159    13.5
160    13.5
161    13.5
162    13.5
163    13.5
164    13.5
165    13.5
Name: GSM, dtype: float64

### Convert Final Data to Excel

#### Group by Grade

In [34]:
# Membaca file Excel
df = df_final

# Memisahkan data berdasarkan awalan pada kolom Grade
df_toilet = df[df['Grade'].str.startswith('T', na=False) & 
               ~df['Grade'].str.startswith('TW', na=False)]
df_towel = df[df['Grade'].str.startswith('TW', na=False)]
df_facial = df[df['Grade'].str.startswith('FC', na=False)]

# Menampilkan jumlah data
print("Jumlah data Toilet :", len(df_toilet))
print("Jumlah data Towel  :", len(df_towel))
print("Jumlah data Facial :", len(df_facial))

Jumlah data Toilet : 106
Jumlah data Towel  : 85
Jumlah data Facial : 0


Simpan ke file Excel terpisah

In [35]:
df.to_excel('Final_PM17.xlsx', index=False)

In [36]:
df_toilet.to_excel("Final_PM17-Toilet.xlsx", index=False)
#df_towel.to_excel("Final_PM17-Towel.xlsx", index=False)
#df_facial.to_excel("Final_PM17-Facial.xlsx", index=False)